# Scriptor: train the handwriting AI (base model)

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

You need the three IAM-OnDB files in Google Drive, in a folder named `IAM-OnDB`:
`lineStrokes-all.tar.gz`, `ascii-all.tar.gz`, `original-xml-all.tar.gz`.

The trained model is saved to `Drive/Scriptor/graves_base.pt` every time it improves,
so if Colab disconnects you keep the best model so far.

In [ ]:
# 1. Connect Google Drive and check the GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Download Scriptor
!rm -rf /content/scriptor && git clone -q https://github.com/Machahary07/scriptor.git /content/scriptor
!pip -q install "Hershey-Fonts>=2.1"
%cd /content/scriptor

In [ ]:
# 3. Unpack IAM-OnDB from Drive
import glob, os, tarfile
src = '/content/drive/MyDrive/IAM-OnDB'
dst = '/content/iam'
os.makedirs(dst, exist_ok=True)
for name in ['lineStrokes-all.tar.gz', 'ascii-all.tar.gz', 'original-xml-all.tar.gz']:
    path = os.path.join(src, name)
    assert os.path.exists(path), f'Missing {path}. Upload it to Drive/IAM-OnDB first.'
    print('unpacking', name)
    with tarfile.open(path) as t:
        t.extractall(dst)
# the loader expects dst/lineStrokes, dst/ascii and dst/original
for want in ['lineStrokes', 'ascii', 'original']:
    if not os.path.isdir(os.path.join(dst, want)):
        found = [d for d in glob.glob(dst + '/**/' + want, recursive=True) if os.path.isdir(d)]
        assert found, f'could not find a {want} folder inside the archives'
        os.symlink(found[0], os.path.join(dst, want))
print({d: len(glob.glob(f'{dst}/{d}/**/*.*', recursive=True)) for d in ['lineStrokes', 'ascii', 'original']})

In [ ]:
# 4. Quick check: 1 epoch on 200 lines (about a minute)
!python training/pretrain.py --data /content/iam --limit 200 --epochs 1 --out /content/check.pt

In [ ]:
# 5. Full training (2-4 hours). Best model is saved to Drive whenever it improves.
!mkdir -p /content/drive/MyDrive/Scriptor
!python training/pretrain.py --data /content/iam --epochs 20 --patience 4 --out /content/drive/MyDrive/Scriptor/graves_base.pt

## Done
Download `Drive/Scriptor/graves_base.pt` and put it in `scriptor/data/models/` on your laptop.
Then in the app: **Writers -> Train LoRA adapter** to fine-tune it on your own handwriting.